# Sentiment prediction of kindle reviews

##### Simple workflow
1. [X] Preprocessing and cleaning the dataset
2. [X] Train Test split
3. [X] Word embeddings
4. [X] Training
5. [X] Testing

### Preprossing the data

In [29]:
import pandas as pd
review = pd.read_csv('all_kindle_review.csv')
review.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [30]:
#We only require rating and review text for this specific model so we will drop everything else
review = review[['reviewText', 'rating']]
review.head() 

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [31]:
review.shape

(12000, 2)

In [32]:
review.isnull().sum() #checking for null values

reviewText    0
rating        0
dtype: int64

In [33]:
review['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [34]:
review['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [35]:
review['rating'] = review['rating'].apply(lambda x:0 if x<3 else 1) #Positive review is 1, negative is 0

In [36]:
review['reviewText'] = review['reviewText'].str.lower()

In [37]:
type(review)

pandas.core.frame.DataFrame

In [38]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
corpus = []

for i in range(0, len(review)):
    X = re.sub('^[a-zA-Z0-9]', ' ', review['reviewText'][i])
    X = X.lower()
    X = X.split()
    X = [lemmatizer.lemmatize(word) for word in X if not word in set(stopwords.words('english'))]
    X = ' '.join(X)
    corpus.append(X)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\goggl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [39]:
type(corpus)

list

### Train Test Split

In [40]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    corpus,
    review['rating'],
    random_state=67,
    test_size=0.20
)

### Word embeddings

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

In [42]:
x_train = tfidf.fit_transform(x_train).toarray()
x_test = tfidf.fit_transform(x_test).toarray()

### Traing Naive Bayes model

In [45]:
from sklearn.naive_bayes import GaussianNB
model_nb = GaussianNB().fit(x_train, y_train)

### Testing

In [46]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [47]:
y_pred = model_nb.predict(x_test)

In [48]:
accuracy_score(y_pred=y_pred, y_true= y_test)

0.60625

In [49]:
print(confusion_matrix(y_pred=y_pred, y_true=y_test))

[[ 105  691]
 [ 254 1350]]


In [50]:
print(classification_report(y_pred=y_pred, y_true=y_test))

              precision    recall  f1-score   support

           0       0.29      0.13      0.18       796
           1       0.66      0.84      0.74      1604

    accuracy                           0.61      2400
   macro avg       0.48      0.49      0.46      2400
weighted avg       0.54      0.61      0.56      2400

